# Run Josh's experiment with attention masking

Choose an **L4 or A100 GPU**, then **Run all**. This launcher downloads the masking branch and executes the existing `colab.ipynb` runner with the preset below. It loads the archived run, runs the checks and smoke pass, then completes Part 0, H1, H2 and the new H1 masking experiment. Previously saved answers are skipped.

The default runs the full benchmark with Qwen3-4B/bf16 across seeds `[0, 1, 2]`, with no per-cell question limit. The existing seed-0 archive is loaded first; Part 0 completes the missing probes before the full masking stage. H1 and H2 also expand to their full three-seed designs.

Masking uses the two-fact toy and two-hop MuSiQue under AA/AB/BA/BB, filtered by the full Part 0 composition labels. Its eligible count is printed after those labels are available. For reference, the original seed-0/100-per-cell sample had 740 eligible question rows (640 for the distractor control); the full count must be computed from the new labels.

To reproduce only Josh's original sample plus masking, set `SEEDS = [0]` and `N_PER_CELL = 100`. Smoke always uses the first seed and two unfiltered questions per cell; extra smoke rows outside the composition subset do not enter full-run tables.

The source defaults to the published fork branch so Josh can run before merging the pull request. After merging, `REPO` can be changed to `https://github.com/JoshBong/latent-dispatch.git` and `COMMIT` to `main`. The exact resolved commit is printed and saved with results. The checkout and results live in Google Drive; authorize Drive when prompted. Run all again with the same `ROOT` to resume across runtimes. An existing checkout keeps its commit, so choose a new `ROOT` when changing source or code version.

Raw results and executed notebooks persist under `ROOT` in Drive. Download `/content/results-Qwen3-4B-bfloat16.tar.gz` when finished for a single shareable bundle of previous and new rows, masking manifests, and executed experiment notebooks. Keep the browser tab open while running.

In [ ]:
REPO = "https://github.com/ajit-mallav/latent-dispatch.git"
COMMIT = "masking-experiment"
ROOT = "/content/drive/MyDrive/latent-dispatch-masking"
MODEL = "Qwen/Qwen3-4B"
DTYPE = "bfloat16"
SEEDS = [0, 1, 2]
N_PER_CELL = None
LOAD = ["Qwen3-4B-bfloat16-2026-11-13"]
NOTEBOOKS = ["part0_ruler", "h1_encoding", "h2_thinking", "h1_masking"]
SKIP = {"h2_thinking": ["t1_1024"]}  # same omitted text-report control as Josh's original run
RUN_TESTS = True
GATE_BOXED, GATE_CUT = 50.0, 20.0

In [ ]:
import os
from pathlib import Path
import subprocess
import sys

import torch

assert torch.cuda.is_available(), "Select an L4 or A100 GPU runtime before running this notebook"
if DTYPE == "bfloat16" and not torch.cuda.is_bf16_supported():
    raise RuntimeError("This GPU does not support bf16; select an L4 or A100")
if ROOT.startswith("/content/drive/"):
    from google.colab import drive
    drive.mount("/content/drive")
checkout = Path(ROOT)
if not checkout.exists():
    subprocess.run(["git", "clone", REPO, ROOT], check=True)
    subprocess.run(["git", "-C", ROOT, "checkout", COMMIT], check=True)
else:
    origin = subprocess.check_output(["git", "-C", ROOT, "remote", "get-url", "origin"], text=True).strip()
    actual = subprocess.check_output(["git", "-C", ROOT, "rev-parse", "HEAD"], text=True).strip()
    wanted = subprocess.check_output(["git", "-C", ROOT, "rev-parse", f"{COMMIT}^{{commit}}"], text=True).strip()
    if origin != REPO or actual != wanted:
        raise RuntimeError("ROOT contains another source checkout; choose a new ROOT")
if not (checkout / "notebooks/h1_masking.ipynb").exists():
    raise RuntimeError("This checkout predates masking; choose a new ROOT")
os.chdir(ROOT)
subprocess.run(["git", "log", "-1", "--format=running commit %h (%s)"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "papermill>=2.6"], check=True)

In [ ]:
import papermill as pm

runner_output = checkout / "notebooks/runs/masking-runner.ipynb"
runner_output.parent.mkdir(parents=True, exist_ok=True)
pm.execute_notebook(
    "colab.ipynb", str(runner_output), kernel_name="python3", log_output=True,
    parameters=dict(REPO=REPO, COMMIT=COMMIT, ROOT=ROOT, MODEL=MODEL, DTYPE=DTYPE,
                    SEEDS=SEEDS, N_PER_CELL=N_PER_CELL, LOAD=LOAD, NOTEBOOKS=NOTEBOOKS,
                    SKIP=SKIP, RUN_TESTS=RUN_TESTS, GATE_BOXED=GATE_BOXED, GATE_CUT=GATE_CUT))

The common runner prints per-arm coverage, output-quality audits, paired EM and gold-answer likelihood comparisons. The Drive checkout retains raw results and executed notebooks. A stopped run resumes from saved question rows on the same or a new runtime. Download the runtime-local results tarball from Colab's Files panel before deleting the runtime if you want a single bundle to share.